In [4]:
import pandas as pd
import glob
import os

csv_folder = 'Championship21-25/'
all_files = glob.glob(os.path.join(csv_folder, '*.csv'))

In [6]:
print(f'Found {len(all_files)} CSV files.')

Found 4 CSV files.


In [8]:
essential_cols = [
    'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR',
    'AvgH', 'AvgD', 'AvgA'
]

In [10]:
data_list = []

for file in all_files:
    df = pd.read_csv(file, usecols = lambda x: x in essential_cols)
    df.columns = df.columns.str.strip()
    df['Date'] = pd.to_datetime(df['Date'], dayfirst = True, errors = 'coerce')
    df.dropna(subset = ['Date', 'HomeTeam', 'AwayTeam', 'FTR'])
    data_list.append(df)

all_data = pd.concat(data_list, ignore_index = True)



In [12]:
all_data.sort_values(['Div', 'Date'], inplace = True)
all_data.reset_index(drop = True, inplace = True)
all_data.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,AvgH,AvgD,AvgA
0,E1,2021-08-06,19:45,Bournemouth,West Brom,2,2,D,1,1,D,2.31,3.32,3.14
1,E1,2021-08-07,15:00,Blackburn,Swansea,2,1,H,1,0,H,2.14,3.29,3.53
2,E1,2021-08-07,15:00,Bristol City,Blackpool,1,1,D,1,0,H,2.22,3.32,3.30
3,E1,2021-08-07,15:00,Cardiff,Barnsley,1,1,D,0,0,D,2.28,3.18,3.31
4,E1,2021-08-07,15:00,Derby,Huddersfield,1,1,D,1,1,D,4.64,3.73,1.75


In [14]:
import numpy as np

teams = pd.unique(all_data[['HomeTeam', 'AwayTeam']].values.ravel())
priors = {}

In [16]:
for team in teams:
    home_games = all_data[all_data['HomeTeam'] == team]
    away_games = all_data[all_data['AwayTeam'] == team]
    total_games = len(home_games) + len(away_games)
    home_wins = (home_games['FTR'] == 'H').sum()
    away_wins = (away_games['FTR'] == 'A').sum()
    total_wins = home_wins + away_wins
    prior = (total_wins + 1) / (total_games + 2)
    priors[team] = prior
priors_df = pd.DataFrame(list(priors.items()), columns=['Team', 'PriorWin'])
priors_df.head()
    

,Team,PriorWin
0,Bournemouth,0.541667
1,West Brom,0.392473
2,Blackburn,0.392473
3,Swansea,0.360215
4,Bristol City,0.349462


In [18]:
all_data['Date'] = pd.to_datetime(all_data['Date'], dayfirst=True)

In [22]:
all_data = all_data.sort_values('Date').reset_index(drop=True)

all_data.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,AvgH,AvgD,AvgA
0,E1,2021-08-06,19:45,Bournemouth,West Brom,2,2,D,1,1,D,2.31,3.32,3.14
1,E1,2021-08-07,20:00,Sheffield United,Birmingham,0,1,A,0,1,A,1.82,3.45,4.64
2,E1,2021-08-07,15:00,QPR,Millwall,1,1,D,1,1,D,2.10,3.33,3.60
3,E1,2021-08-07,15:00,Preston,Hull,1,4,A,1,1,D,2.34,3.27,3.11
4,E1,2021-08-07,15:00,Luton,Peterboro,3,0,H,1,0,H,2.21,3.42,3.25


In [24]:

elo_ratings = {team: 1500 for team in teams}
K = 100

def elo_expected(home_elo, away_elo):
    return 1 / (1 + 10 ** ((away_elo - home_elo)/ 400))

In [26]:
elo_history = []
for idx, row in all_data.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    ftr = row['FTR']
    home_elo = elo_ratings[home]
    away_elo = elo_ratings[away]
    expected_home = elo_expected(home_elo, away_elo)
    expected_away = 1- expected_home 
    if ftr == 'H':
        score_home, score_away = 1, 0
    elif ftr == 'A':
        score_home, score_away = 0, 1
    else:
        score_home, score_away = 0.5, 0.5
    elo_ratings[home] += K * (score_home - expected_home)
    elo_ratings[away] += K * (score_away - expected_away)
    elo_history.append({
        'Date': row['Date'],
        'HomeTeam': home,
        'AwayTeam': away,
        'HomeElo': home_elo,
        'AwayElo': away_elo,
        'FTR': ftr
    })

elo_df = pd.DataFrame(elo_history)
elo_df.head()




,Date,HomeTeam,AwayTeam,HomeElo,AwayElo,FTR
0,2021-08-06,Bournemouth,West Brom,1500.0,1500.0,D
1,2021-08-07,Sheffield United,Birmingham,1500.0,1500.0,A
2,2021-08-07,QPR,Millwall,1500.0,1500.0,D
3,2021-08-07,Preston,Hull,1500.0,1500.0,A
4,2021-08-07,Luton,Peterboro,1500.0,1500.0,H


In [28]:
#We now merge the Elo data back into all_data
all_data['Date'] = pd.to_datetime(all_data['Date'], dayfirst = True)
elo_df['Date'] = pd.to_datetime(elo_df['Date'], dayfirst = True)

df = all_data.merge(
    elo_df,
    on = ['Date', 'HomeTeam', 'AwayTeam', 'FTR'],
    how = 'left'
)
df[['HomeTeam','AwayTeam','HomeElo','AwayElo']]

,HomeTeam,AwayTeam,HomeElo,AwayElo
0,Bournemouth,West Brom,1500.000000,1500.000000
1,Sheffield United,Birmingham,1500.000000,1500.000000
2,QPR,Millwall,1500.000000,1500.000000
3,Preston,Hull,1500.000000,1500.000000
4,Luton,Peterboro,1500.000000,1500.000000
...,...,...,...,...
2203,Burnley,Millwall,1842.802367,1655.058400
2204,Bristol City,Preston,1567.542326,1339.507480
2205,Watford,Sheffield Weds,1330.441854,1425.921385
2206,Norwich,Cardiff,1376.954298,1381.527849


In [30]:
#Convert the bookmakers odds into implied probabilities 
def implied_probs(row):
    odds = np.array([row['AvgH'], row['AvgD'], row['AvgA']])
    inv = 1 / odds
    return inv / inv.sum()

market_probs = df.apply(implied_probs, axis = 1, result_type = 'expand')
market_probs.columns = ['Mkt_H','Mkt_D','Mkt_A']
df = pd.concat([df, market_probs], axis = 1)
df[['Mkt_H','Mkt_D','Mkt_A']].head()

,Mkt_H,Mkt_D,Mkt_A
0,0.411277,0.286160,0.302564
1,0.520894,0.274790,0.204316
2,0.451679,0.284842,0.263479
3,0.405185,0.289949,0.304866
4,0.429886,0.277792,0.292322


In [32]:
elo_all = np.concatenate([df['HomeElo'], df['AwayElo']])
elo_mean = elo_all.mean()
elo_std = elo_all.std()

#calculate z statistic
df['HomeElo_z'] = (df['HomeElo']-elo_mean) / elo_std
df['AwayElo_z'] = (df['AwayElo']-elo_mean) / elo_std

In [34]:
df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AvgH,AvgD,AvgA,HomeElo,AwayElo,Mkt_H,Mkt_D,Mkt_A,HomeElo_z,AwayElo_z
0,E1,2021-08-06,19:45,Bournemouth,West Brom,2,2,D,1,1,...,2.31,3.32,3.14,1500.0,1500.0,0.411277,0.286160,0.302564,0.006778,0.006778
1,E1,2021-08-07,20:00,Sheffield United,Birmingham,0,1,A,0,1,...,1.82,3.45,4.64,1500.0,1500.0,0.520894,0.274790,0.204316,0.006778,0.006778
2,E1,2021-08-07,15:00,QPR,Millwall,1,1,D,1,1,...,2.10,3.33,3.60,1500.0,1500.0,0.451679,0.284842,0.263479,0.006778,0.006778
3,E1,2021-08-07,15:00,Preston,Hull,1,4,A,1,1,...,2.34,3.27,3.11,1500.0,1500.0,0.405185,0.289949,0.304866,0.006778,0.006778
4,E1,2021-08-07,15:00,Luton,Peterboro,3,0,H,1,0,...,2.21,3.42,3.25,1500.0,1500.0,0.429886,0.277792,0.292322,0.006778,0.006778


In [36]:
elo_all = np.concatenate(df['HomoeElo'], df['AwayElo'])
elo_mean

array([1500.        , 1500.        , 1500.        , ..., 1425.92138487,
       1381.52784923, 1620.53609519])